# 프로젝트 1 - Weekend 3: 메모리와 프롬프트 엔지니어링

| 항목 | 내용 |
|------|------|
| **프로젝트** | 주택청약 FAQ 챗봇 - 최종 완성 |
| **핵심 기술** | Few-shot, CoT, Memory, Gradio |

## 환경 설정

In [ ]:
# !pip install -q openai langchain-openai langchain-community langchain faiss-cpu python-dotenv gradio

In [ ]:
# colab 사용 시 아래 주석 해제
# !pip install -q openai langchain-openai langchain-community langchain faiss-cpu python-dotenv gradio
# import os
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
# 환경 설정
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS

client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ 환경 설정 완료")

## 📦 실습용 샘플 데이터

In [ ]:
# ============================================================
# 주택청약 FAQ 샘플 데이터 (실습용)
# ============================================================
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장",
     "question": "주택청약종합저축이란 무엇인가요?",
     "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨",
     "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
    {"id": "FAQ004", "category": "청약통장",
     "question": "청약통장 1순위 조건은 무엇인가요?",
     "answer": "1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입",
     "keywords": ["1순위", "가입기간", "예치금", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ005", "category": "청약자격",
     "question": "주택 청약 신청 자격 조건은 무엇인가요?",
     "answer": "1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능",
     "keywords": ["청약자격", "만19세", "무주택", "세대주"], "difficulty": "easy"},
    {"id": "FAQ006", "category": "청약자격",
     "question": "무주택자 기준은 무엇인가요?",
     "answer": "본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함",
     "keywords": ["무주택", "세대원", "소형주택", "분양권"], "difficulty": "medium"},
    {"id": "FAQ009", "category": "특별공급",
     "question": "특별공급의 종류에는 어떤 것이 있나요?",
     "answer": "1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대",
     "keywords": ["특별공급", "기관추천", "다자녀", "신혼부부", "생애최초"], "difficulty": "medium"},
    {"id": "FAQ010", "category": "특별공급",
     "question": "신혼부부 특별공급 조건은 무엇인가요?",
     "answer": "1) 혼인기간 7년 이내 무주택 세대주\n2) 소득: 도시근로자 월평균소득 100~140%\n3) 전용면적 85㎡ 이하\n4) 혼인기간 짧을수록 + 자녀 많을수록 가점 높음\n5) 예비 신혼부부도 신청 가능",
     "keywords": ["신혼부부", "혼인기간", "소득기준", "가점"], "difficulty": "medium"},
    {"id": "FAQ013", "category": "일반공급",
     "question": "가점제와 추첨제의 차이는 무엇인가요?",
     "answer": "가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%",
     "keywords": ["가점제", "추첨제", "84점", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ017", "category": "당첨/계약",
     "question": "당첨자 발표는 어떻게 확인하나요?",
     "answer": "1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인",
     "keywords": ["당첨자발표", "청약홈", "SMS알림", "서류제출"], "difficulty": "easy"},
    {"id": "FAQ020", "category": "당첨/계약",
     "question": "재당첨 제한이란 무엇인가요?",
     "answer": "당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)",
     "keywords": ["재당첨제한", "10년", "7년", "세대원"], "difficulty": "medium"},
    {"id": "FAQ023", "category": "기타",
     "question": "청약홈 사이트는 어떻게 이용하나요?",
     "answer": "청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공",
     "keywords": ["청약홈", "공인인증서", "간편인증", "가점계산"], "difficulty": "easy"},
]

SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
]

print(f"📦 FAQ 데이터 로드 완료: {len(SAMPLE_FAQ_DATA)}개 QA, {len(SAMPLE_TEST_QUERIES)}개 테스트 질의")

## 🔧 Weekend 2 복원

In [ ]:
# Weekend 2 복원: 벡터 스토어 + RAG 체인
documents = [Document(
    page_content=f"질문: {f['question']}\n답변: {f['answer']}",
    metadata={"id": f["id"], "category": f["category"]}
) for f in SAMPLE_FAQ_DATA]

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n---\n".join([f"[{d.metadata.get('category','')}] {d.page_content}" for d in docs])

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "주택청약 전문 상담원입니다. 참고 FAQ:\n{context}\n\nFAQ 기반으로 친절하게 답변하세요."),
    ("user", "{question}")
])

baseline_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print(f"✅ RAG 파이프라인 복원 완료 ({len(documents)}개 문서)")
print(f"테스트: {baseline_chain.invoke('청약통장이 뭐예요?')[:80]}...")

---
## 사이클 1: Weekend 2 복원 + 기준선 측정

Weekend 2의 RAG 체인을 복원하고, `SAMPLE_TEST_QUERIES` 5개로 응답 시간과 답변 길이의 기준선을 측정하세요.

In [ ]:
# 사이클 1
import time


---
## 사이클 2: Few-shot 프롬프트

FAQ 답변 예시 3개를 포함한 few-shot 프롬프트를 만들고, 기본 프롬프트 대비 답변 형식이 개선되는지 비교하세요.

In [ ]:
# 사이클 2


---
## 사이클 3: Chain-of-Thought 프롬프트

"1단계-문제 파악, 2단계-원인 분석, 3단계-해결 방법" 사고 과정을 명시하는 CoT 프롬프트를 만들고, 복잡한 질문 3개로 테스트하세요.

In [ ]:
# 사이클 3


---
## 사이클 4: ConversationBufferMemory

`ConversationBufferMemory`와 `MessagesPlaceholder`를 사용해 이전 대화를 기억하는 챗봇을 만드세요. 3턴 이상의 연속 대화를 테스트하세요.
`ConversationBufferMemory`가 deprecated된 `Modern Langchain`을 활용해서도 구현해보세요

In [ ]:
# 사이클 4
from langchain.memory import ConversationBufferMemory


---
## 사이클 5: ConversationBufferWindowMemory

`ConversationBufferWindowMemory(k=3)`으로 최근 3턴만 기억하는 메모리를 만들고, 5턴 대화 후 초기 대화가 잊혀지는지 확인하세요.
`ConversationBufferWindowMemory`가 deprecated된 `Modern Langchain`을 활용해서도 구현해보세요

In [ ]:
# 사이클 5
from langchain.memory import ConversationBufferWindowMemory


---
## 사이클 6: RAG + Memory 통합 챗봇

RAG 검색 + 대화 메모리를 결합한 `FAQChatbotV3` 클래스를 만드세요. `ask(question)` 메서드와 `reset()` 메서드를 구현하고, 5턴 멀티턴 대화를 테스트하세요.

In [ ]:
# 사이클 6
import time


---
## 사이클 7: 의도 분류기

사용자 메시지를 greeting/question/complaint/chitchat으로 분류하는 체인을 만들고, 의도별로 다른 응답 전략을 적용하세요. 5가지 메시지로 테스트하세요.

In [ ]:
# 사이클 7
import json


---
## 사이클 8: 답변 불가 처리

`similarity_search_with_score`의 거리 값으로 FAQ 범위 밖 질문을 감지하세요. 답변 가능/불가를 판별하는 `smart_answer(question)` 함수를 만들고, FAQ 관련/무관 질문 각 3개로 테스트하세요.

In [ ]:
# 사이클 8


---
## 사이클 9: Gradio 최종 UI

의도 분류 + RAG + 메모리를 통합한 최종 `gr.ChatInterface`를 만드세요.

In [ ]:
# 사이클 9
import gradio as gr


---
## 사이클 10: 최종 데모 & 프로젝트 회고

5턴 멀티턴 대화 데모를 실행하고, 10개 질문으로 최종 벤치마크를 돌리세요. 3주간 구현한 기능 목록과 향후 개선점 3가지를 정리하세요.

In [ ]:
# 사이클 10
import time


---
## 프로젝트 1 완료! 🎉

3주간 수고하셨습니다!